# Lecture 21: ANOVA and Linear Regression
**Ling 250/450: Data Science for Linguistics** | Spring 2026

## Overview

This notebook accompanies the slides on ANOVA and linear regression. Rather than re-explaining the concepts from the slides, it focuses on working through examples in R and building intuition for how to interpret the output.

Sections:

1. **ANOVA**: comparing means across two or more groups
   - The F-statistic and F-distribution
   - Running `aov()` and reading `summary()`
   - Multiple comparisons and the Holm correction
2. **Linear regression**: modeling a continuous outcome from predictors
   - Error minimization and the best-fit line
   - Multivariable and categorical predictors
   - Running `lm()` and interpreting the coefficient table

In [ ]:
library(dplyr)

---

## Part 1: ANOVA

**ANOVA** stands for **Analysis of Variance**. Like t-tests, it asks whether samples share the same population mean. Unlike t-tests, it handles **two or more groups** simultaneously.

**Hypotheses:**

- $H_0$: $\mu_1 = \mu_2 = \ldots = \mu_G$ (all group means are equal)
- $H_1$: Not $(\mu_1 = \mu_2 = \ldots = \mu_G)$ (at least one mean differs)

Note: $H_1$ is *not* the same as saying all means are different from each other — it just says they aren't all equal.

ANOVA compares two sources of variation:

- **Between-group variation**: how spread apart are the group means?
- **Within-group variation**: how spread out are observations within each group?

The key intuition: the larger the between-group variation is *relative to* the within-group variation, the more confident we can be that the groups genuinely differ.

In [ ]:
# Visualizing between-group vs within-group variation
x <- seq(-2, 9, length.out = 500)
group_means <- c(1, 4, 7)

par(mfrow = c(2, 1), mar = c(3, 1, 3, 1))

# Top: narrow within-group spread — means are easy to distinguish
sigma_narrow <- 0.8
plot(NULL, xlim = c(-2, 9), ylim = c(0, 0.55), xaxt = "n", yaxt = "n",
     xlab = "", ylab = "",
     main = "Between-group variation large relative to within-group")
for (mu in group_means) lines(x, dnorm(x, mu, sigma_narrow), lwd = 2)
axis(1, at = group_means, labels = c("group 1", "group 2", "group 3"))

# Bottom: wide within-group spread — means are harder to distinguish
sigma_wide <- 2.2
plot(NULL, xlim = c(-2, 9), ylim = c(0, 0.22), xaxt = "n", yaxt = "n",
     xlab = "", ylab = "",
     main = "Within-group variation large relative to between-group")
for (mu in group_means) lines(x, dnorm(x, mu, sigma_wide), lwd = 2)
axis(1, at = group_means, labels = c("group 1", "group 2", "group 3"))

par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1)

### The F-statistic

ANOVA summarizes the comparison as a single number called the **F-statistic**:

$$F = \frac{MS_b}{MS_w}$$

where $MS_b$ is the **mean squared between-group variation** and $MS_w$ is the **mean squared within-group variation** (the residuals). We're purposefully glossing over how these are computed — the key point is:

- **Large F** → between-group variation dominates → low p-value
- **F near 1** → groups are about as different as you'd expect from noise → high p-value

Significance comes from the **F-distribution**, which is right-skewed and bounded at 0. Its exact shape depends on two degree-of-freedom parameters: `df1` (number of groups minus 1) and `df2` (total observations minus number of groups). The humped shape appears when `df1 > 2`:

In [ ]:
# The F-distribution shape depends on df1 and df2.
# With df1 = 2 (3 groups) the distribution is monotonically decreasing from 0;
# the characteristic hump appears when df1 > 2.
# We use df1 = 5 here to show the typical shape.
curve(
  df(x, df1 = 5, df2 = 30), from = 0, to = 6, n = 500,
  xlab = "Observed Value", ylab = "Probability Density",
  main = "F-distribution (illustrative: df1 = 5, df2 = 30)",
  col = "#444488", lwd = 2
)

### Example: Clinical Trial

We'll use a dataset from a clinical trial testing two antidepressant drugs and a placebo, measuring mood improvement:

- **Predictor** (grouping variable): `drug` — `placebo`, `anxifree`, `joyzepam`
- **Outcome**: `mood.gain` — continuous measure of mood improvement

Research question: **does drug type affect mood gain?**

In [ ]:
clin_trial <- data.frame(
  drug = factor(
    c(rep("placebo", 6), rep("anxifree", 6), rep("joyzepam", 6)),
    levels = c("placebo", "anxifree", "joyzepam")
  ),
  therapy = factor(rep(c(rep("no.therapy", 3), rep("CBT", 3)), 3)),
  mood.gain = c(
    0.5, 0.3, 0.1,   # placebo, no therapy
    0.6, 0.9, 0.3,   # placebo, CBT
    0.6, 0.4, 0.2,   # anxifree, no therapy
    1.1, 0.8, 1.2,   # anxifree, CBT
    1.4, 1.7, 1.3,   # joyzepam, no therapy
    1.8, 1.3, 1.4    # joyzepam, CBT
  )
)
clin_trial

In [ ]:
# Visualize the groups before running any test
boxplot(
  mood.gain ~ drug, data = clin_trial,
  xlab = "Drug", ylab = "Mood Gain",
  main = "Mood Gain by Drug Condition",
  col = c("gray80", "steelblue", "salmon")
)

### Running `aov()`

The R function for ANOVA is `aov()`. It takes a **formula** argument — `outcome ~ group_variable` — and a `data` argument. It returns a complex R object:

In [ ]:
my_anova <- aov(mood.gain ~ drug, data = clin_trial)

`print()` gives a basic summary, but the **Residuals** row is the main thing to note here — it shows the variability *not* explained by drug (i.e., within-group variation):

In [ ]:
print(my_anova)

For the p-value and F-statistic, use `summary()`:

In [ ]:
summary(my_anova)

### Interpreting the Result

The `summary()` output gives:

| Column | Meaning |
|--------|---------|
| `Df` | Degrees of freedom — for `drug`: number of groups minus 1 |
| `Sum Sq` | Sum of squared deviations |
| `Mean Sq` | Sum Sq / Df (i.e., $MS_b$ for drug, $MS_w$ for residuals) |
| `F value` | $MS_b / MS_w$ |
| `Pr(>F)` | p-value: probability of this F value or higher under $H_0$ |

The `***` significance code means p < 0.001 — very strong evidence against $H_0$.

**Important caveat:** rejecting $H_0$ only tells us that the group means aren't all equal. It doesn't tell us *which* pairs of groups are different. The null is $\mu_P = \mu_A = \mu_J$; the alternative is just "not all equal."

The null can be rephrased as: "drug has no effect on mood." The ANOVA tells us drug *does* have an effect — but not which drug is responsible.

### Multiple Comparisons

With 3 groups, there are 8 possible patterns of which means are equal to which. ANOVA collapses 7 of them into its alternative hypothesis. To get pairwise answers — does placebo differ from anxifree? does joyzepam differ from anxifree? — we can run **pairwise t-tests**:

In [ ]:
pairwise.t.test(
  x = clin_trial$mood.gain,   # outcome variable
  g = clin_trial$drug,        # grouping variable
  p.adjust.method = "none"    # no correction — see below
)

The matrix shows p-values for each pair. But there's a problem: when we run many tests, some will be significant **by chance**. At $\alpha = 0.05$, each test has a 5% false positive rate — run enough tests and you'll almost certainly get a spurious result. This is a form of **p-value fishing** (and the same post-hockery problem from the readings).

The solution: **adjust p-values** to account for the fact that multiple tests are being run. The **Holm correction** scales p-values as if the whole set of tests were one joint test:

In [ ]:
pairwise.t.test(
  x = clin_trial$mood.gain,
  g = clin_trial$drug,
  p.adjust.method = "holm"
)

After correction, p-values are generally larger (more conservative). The joyzepam comparisons remain significant, but the anxifree vs placebo comparison is still not.

Why not just use corrected pairwise tests instead of ANOVA at all? Because **ANOVA encourages well-formulated hypotheses**: you decide in advance that you're asking whether the groups differ as a whole, rather than fishing for which specific pairs happen to be significant.

### ANOVA Assumptions

1. **Independence**: observations are independent — e.g., the same participant isn't in two groups
2. **Homogeneity of variance**: all groups have the same variance. If this is violated, use the **Welch one-way test**: `oneway.test(formula, data = df)`
3. **Normality**: each group is approximately Normally distributed. This is a strong assumption; there are methods to check it, but we won't cover them here.

> **Connection to t-tests**: with exactly two groups, ANOVA is mathematically equivalent to the Student's t-test. The Student's t-test can be thought of as a special case of ANOVA.

---

## Part 2: Linear Regression

Linear regression **models a continuous outcome** (the *response* or *dependent variable*) as a linear function of one or more *predictors* (or *independent variables*).

The basic model with one predictor:

$$y = \beta x + \alpha$$

This is the same as $y = mx + b$ from algebra:

- $\alpha$ (alpha) — **intercept/bias**: predicted value of $y$ when $x = 0$
- $\beta$ (beta) — **slope**: how much $y$ changes for each 1-unit increase in $x$

Both are **learned coefficients** — the model estimates them from the data.

"Linear" refers to the fact that the effects of predictors are **summed together**.

### Error Minimization

The model isn't perfect — observed data points don't fall exactly on the line. The coefficients are chosen to **minimize the residual error** $\epsilon$:

$$y = \beta x + \alpha + \epsilon$$

where $\epsilon$ is the gap between the predicted line and each observed point. Specifically, the model minimizes the **sum of squared residuals** — which yields the **best fit line**.

In [ ]:
# Generate some synthetic data to illustrate the best fit line
set.seed(12)
x_demo <- seq(0, 9, length.out = 10)
y_demo <- 2.5 * x_demo + 15 + rnorm(10, mean = 0, sd = 4)

lm_demo <- lm(y_demo ~ x_demo)
y_pred  <- predict(lm_demo)

plot(
  x_demo, y_demo,
  pch = 19, col = "#222266",
  xlab = "X", ylab = "Y",
  main = "Best fit line and residuals",
  ylim = c(0, 45)
)
abline(lm_demo, col = "firebrick", lwd = 2)
# Draw residual segments from each point to the fitted line
segments(x_demo, y_demo, x_demo, y_pred, col = "#222266", lwd = 1.5)

### Multivariable Regression

The formula generalizes to **multiple predictors**:

$$y = \alpha + \beta_1 x_1 + \beta_2 x_2 + \ldots + \epsilon$$

In matrix notation: $Y = X\beta + \epsilon$. The fitting procedure is the same.

In R:

```r
model <- lm(y ~ predictor)                   # one predictor
model <- lm(y ~ predictor_1 + predictor_2)   # two predictors
```

### Categorical Predictors: Dummy Variables

A predictor can be **categorical** (a factor) — for example, which of two ponds a fish came from.

Regression software converts factors to **binary "dummy" variables**. For two ponds:

- Pond 1 → $x = 0$ (the **baseline/reference** group)
- Pond 2 → $x = 1$

With this encoding:

- **Pond 1**: $y = \beta \cdot 0 + \alpha = \alpha$ → the intercept is the predicted mean for pond 1
- **Pond 2**: $y = \beta \cdot 1 + \alpha = \alpha + \beta$ → $\beta$ is the *difference* between pond 2 and pond 1

For $n$ categories, $n - 1$ dummy variables are needed (one category is always the baseline). R handles this automatically when the variable is a `factor`.

### R Example: Algae and Fertilizer

We'll model algae growth as a function of two predictors:

- `fertilizer` — continuous, 0–10 scale
- `pond` — categorical, two ponds

The underlying model we'll simulate from is:

$$\text{algae} = 2.0 \cdot \text{fertilizer} + 3.5 \cdot \text{pond2} + \alpha + \epsilon$$

where `pond2` is a dummy variable (1 for pond 2, 0 for pond 1). We want to see whether `lm()` can recover these true coefficients from the data.

In [ ]:
set.seed(3)
n_per_pond <- 12
pond_data <- data.frame(
  fertilizer = round(runif(2 * n_per_pond, min = 0, max = 10), 2),
  pond = factor(c(rep(1, n_per_pond), rep(2, n_per_pond)))
)
pond_data$algae <- round(
  2.0 * pond_data$fertilizer +
  3.5 * (pond_data$pond == 2) +
  rnorm(2 * n_per_pond, mean = 0, sd = 1),
  2
)
head(pond_data, 10)

In [ ]:
pond_model <- lm(algae ~ fertilizer + pond, data = pond_data)

In [ ]:
summary(pond_model)

### Interpreting the Summary

**Coefficients table** — this is the most important part:

| Row | What it estimates |
|-----|-------------------|
| `(Intercept)` | $\alpha$ — predicted algae for **pond 1** when fertilizer = 0 |
| `fertilizer` | $\beta_f$ — algae increase per 1-unit increase in fertilizer (same for both ponds) |
| `pond2` | $\beta_p$ — how much higher pond 2's algae is than pond 1's, at the same fertilizer level |

The `t value` and `Pr(>|t|)` columns test whether each coefficient is **significantly different from zero**. An intercept p-value that isn't significant just means the intercept is close to zero — that's fine and doesn't indicate a problem.

**Bottom section:**

- **Residual standard error**: typical size of prediction errors
- **Multiple R-squared**: fraction of variance in algae explained by the model (0 = nothing, 1 = perfect)
- **F-statistic / p-value**: overall model significance — null hypothesis is that *all* coefficients (except intercept) are zero

The true coefficients were 2.0 (fertilizer) and 3.5 (pond2). Check: how close did `lm()` get?

In [ ]:
# Visualize the data and fitted lines
pond_colors <- c("1" = "steelblue", "2" = "salmon")

plot(
  pond_data$fertilizer, pond_data$algae,
  col  = pond_colors[as.character(pond_data$pond)],
  pch  = 19,
  xlab = "Fertilizer", ylab = "Algae",
  main = "Algae by Fertilizer and Pond"
)
legend("topleft", legend = c("Pond 1", "Pond 2"),
       col = c("steelblue", "salmon"), pch = 19)

# Parallel regression lines: same slope, different intercepts
coefs <- coef(pond_model)
abline(
  a = coefs["(Intercept)"],
  b = coefs["fertilizer"],
  col = "steelblue", lwd = 2
)
abline(
  a = coefs["(Intercept)"] + coefs["pond2"],
  b = coefs["fertilizer"],
  col = "salmon", lwd = 2
)

The two regression lines are **parallel** — that's the key feature of this model. Both ponds have the same slope for fertilizer; pond 2 just has a higher intercept by $\beta_p \approx 3.5$.

> **Note for later**: in the data visualization lecture, you'll see `ggplot` with `geom_smooth(method = "lm")` fit *separate* lines per group — which is a *different* (and more flexible) model that allows different slopes per pond. The parallel-lines model here is appropriate when you believe the slope is the same across groups.

<details>
<summary><b>Challenge:</b> Using the vowel data, fit a linear regression predicting F1 from F2. What does the coefficient on F2 tell you about the relationship between F1 and F2 in the vowel space? Is it significant?</summary>

```r
vowels <- read.csv("../data/vowels.csv", fileEncoding = "UTF-8-BOM")

vowel_model <- lm(F1 ~ F2, data = vowels)
summary(vowel_model)
```

The coefficient on F2 should be negative: higher F2 (front vowels) tends to co-occur with lower F1 (high vowels like /i/), reflecting the structure of the vowel space.
</details>

---

## Quick Reference

### ANOVA

| Task | R code |
|------|---------|
| Run one-way ANOVA | `my_anova <- aov(outcome ~ group, data = df)` |
| Get F-statistic and p-value | `summary(my_anova)` |
| Pairwise t-tests, no correction | `pairwise.t.test(x, g, p.adjust.method = "none")` |
| Pairwise t-tests, Holm correction | `pairwise.t.test(x, g, p.adjust.method = "holm")` |
| Welch one-way test (unequal variances) | `oneway.test(outcome ~ group, data = df)` |

### Linear Regression

| Task | R code |
|------|---------|
| Fit one-predictor model | `model <- lm(y ~ x, data = df)` |
| Fit multiple predictors | `model <- lm(y ~ x1 + x2, data = df)` |
| Summary (coefficients, R², p-values) | `summary(model)` |
| Extract coefficients | `coef(model)` |
| Get predicted values | `predict(model)` |
| Get residuals | `residuals(model)` |

### Key Terms

| Term | Meaning |
|------|---------|
| Predictor / independent variable | Variable(s) used to predict the outcome |
| Response / dependent variable | Variable being predicted |
| Intercept ($\alpha$) | Predicted response when all predictors = 0 |
| Slope ($\beta$) | Change in response per 1-unit increase in predictor |
| Residual ($\epsilon$) | Observed minus predicted (the error) |
| R-squared | Fraction of variance in the response explained by the model |
| Dummy variable | Binary (0/1) encoding of one level of a categorical predictor |
| Baseline/reference group | The category encoded as 0 in dummy coding |